# AIMO3 Math Solver — Kaggle Submission

**Model:** Prefer available `Qwen2.5-Math-72B` variant (`AWQ/GPTQ/fp8`) on H100, else `7B-Instruct`  
**Answer format:** Integer in [0, 99999]  
**Method:** TIR — generate Python → execute → inject output → majority vote  
**Submission:** Kaggle evaluation API (`kaggle_evaluation.aimo_inference_server`)

## Setup checklist
- [ ] GPU accelerator enabled (Settings → Accelerator)
- [ ] Add a Qwen math model from sidebar (`72B` variant preferred, else `7B`)
- [ ] Internet **off** for final submission (internet is disabled during scoring)

## Phases
1. Setup & model loading
2. Integer answer extraction + majority vote
3. TIR solver
4. Benchmark on reference problems
5. Kaggle API submission loop

---
## 1. Setup

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

!pip install -q --no-cache-dir vllm transformers>=4.44.0 sympy pandas tqdm
!rm -rf /root/.cache/pip

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — enable accelerator in notebook Settings")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")

# AIMO3 workhorse setup: optimize around the 7B model first.
# Keep the 72B path available for larger accelerators, but tune defaults for 7B.
if VRAM_GB >= 75:
    MODEL_ID = "Qwen/Qwen2.5-Math-72B-Instruct"
    USE_FP8 = True
    SOLUTIONS_PER_PROBLEM = 8
    MAX_TOKENS = 896
    MAX_ROUNDS = 3
    GENERATION_CHUNK_SIZE = 1
    USE_BEAM_SEARCH = False
    BEAM_WIDTH = 4
    BEAM_BRANCH_FACTOR = 2
else:
    MODEL_ID = "Qwen/Qwen2.5-Math-7B-Instruct"
    USE_FP8 = False
    SOLUTIONS_PER_PROBLEM = 12
    MAX_TOKENS = 768
    MAX_ROUNDS = 4
    GENERATION_CHUNK_SIZE = 2
    USE_BEAM_SEARCH = True
    BEAM_WIDTH = 4
    BEAM_BRANCH_FACTOR = 2

MIN_VALID_ANSWER_COUNT = 2
CODE_TIMEOUT_SECONDS = 10
USE_SCORING_VOTE = True
RUN_BASELINE_COMPARISON = True

# Optional LoRA adapter support (recommended only with 7B base).
USE_LORA_ADAPTER = False
LORA_ADAPTER_PATH = "/kaggle/input/aimo-7b-orm-lora"

os.makedirs("/kaggle/working/results", exist_ok=True)
print(f"Model: {MODEL_ID}")
print(f"Max tokens: {MAX_TOKENS}  |  Solutions per problem: {SOLUTIONS_PER_PROBLEM}")
print(f"Max rounds: {MAX_ROUNDS}  |  Chunk size: {GENERATION_CHUNK_SIZE}")
print(f"Beam search: {USE_BEAM_SEARCH}  |  Width: {BEAM_WIDTH}  |  Branch factor: {BEAM_BRANCH_FACTOR}")
print(f"Min valid answers for vote: {MIN_VALID_ANSWER_COUNT}  |  Scoring vote: {USE_SCORING_VOTE}")
print(f"Run baseline comparison: {RUN_BASELINE_COMPARISON}")
print(f"LoRA enabled: {USE_LORA_ADAPTER}  |  Adapter path: {LORA_ADAPTER_PATH}")


In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer
import glob

# On Kaggle, the model is mounted from "Add data → Models" — no download needed.
# Walk /kaggle/input/ to find the best available weights directory.
def _find_kaggle_model(model_id: str) -> str:
    slug = model_id.split("/")[-1].lower().replace(".", "")
    # Prefer quantized 72B variants first for memory stability on single H100.
    patterns = [
        f"/kaggle/input/*{slug}*awq*",
        f"/kaggle/input/*{slug}*gptq*",
        f"/kaggle/input/*{slug}*",
        f"/kaggle/input/*qwen*math*72b*awq*",
        f"/kaggle/input/*qwen*math*72b*gptq*",
        f"/kaggle/input/*qwen*math*72b*",
        f"/kaggle/input/*qwen*math*7b*",
        f"/kaggle/input/*math*instruct*",
    ]
    for pattern in patterns:
        for hit in sorted(glob.glob(pattern)):
            for root, _, files in os.walk(hit):
                if any(f.endswith(".safetensors") for f in files):
                    return root
    return model_id   # fallback: HuggingFace ID (requires internet ON)

MODEL_PATH = _find_kaggle_model(MODEL_ID)
print(f"Model path: {MODEL_PATH}")

path_l = MODEL_PATH.lower()
if "awq" in path_l:
    QUANTIZATION = "awq"
elif "gptq" in path_l:
    QUANTIZATION = "gptq"
elif USE_FP8:
    QUANTIZATION = "fp8"
else:
    QUANTIZATION = None

IS_72B = "72b" in path_l
MAX_MODEL_LEN = 1536 if IS_72B else 2048
GPU_MEM_UTIL = 0.80 if IS_72B else 0.85

ENABLE_LORA = USE_LORA_ADAPTER and os.path.isdir(LORA_ADAPTER_PATH)
LORA_REQUEST = None
if ENABLE_LORA:
    LORA_REQUEST = LoRARequest("orm_adapter", 1, LORA_ADAPTER_PATH)
else:
    if USE_LORA_ADAPTER:
        print(f"LoRA adapter path not found: {LORA_ADAPTER_PATH}. Continuing without adapter.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

llm = LLM(
    model=MODEL_PATH,
    trust_remote_code=True,
    dtype="auto",
    quantization=QUANTIZATION,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    enable_lora=ENABLE_LORA,
    disable_log_stats=True,
)

print(f"Quantization: {QUANTIZATION}")
print(f"max_model_len={MAX_MODEL_LEN}, gpu_memory_utilization={GPU_MEM_UTIL}")
print(f"LoRA active: {ENABLE_LORA}")
print(f"✓ Model loaded: {MODEL_PATH}")

---
## 2. Answer Extraction & Voting

AIMO3 answers are always integers in [0, 99999]. No LaTeX normalization needed.

We extract integers from `\boxed{}` and fall back to scanning the last lines of the solution.
Majority vote selects the most-agreed-upon integer.

In [ ]:
import re
import subprocess
import sys
from collections import Counter


# ---------------------------------------------------------------------------
# Integer answer extraction
# AIMO3: every answer is an integer in [0, 99999]. No LaTeX normalization needed.
# ---------------------------------------------------------------------------

def extract_integer(text: str) -> int | None:
    """Extract the final integer answer from a solution.

    Priority:
    1. Last \\boxed{N} where N is an integer (handles \\boxed{42}, \\boxed{1234})
    2. Last bare integer on a line that looks like a conclusion
    3. Any integer in [0, 99999] near "answer is" / "= N" patterns
    Returns None if no valid integer found.
    """
    boxed_ints = []
    for m in re.finditer(r'\\boxed\{', text):
        start = m.end()
        depth, i = 1, start
        while i < len(text) and depth > 0:
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
            i += 1
        if depth == 0:
            content = text[start:i - 1].strip()
            content = re.sub(r'\\text\{([^}]*)\}', r'\1', content).strip()
            content_clean = content.replace(',', '').replace(' ', '')
            try:
                val = int(content_clean)
                if 0 <= val <= 99999:
                    boxed_ints.append(val)
            except ValueError:
                pass

    if boxed_ints:
        return boxed_ints[-1]

    patterns = [
        r'(?:answer|result|value)\s*(?:is|=|:)\s*(\d{1,5})\b',
        r'=\s*(\d{1,5})\s*$',
        r'\b(\d{1,5})\s*$',
    ]
    for pat in patterns:
        hits = re.findall(pat, text, re.IGNORECASE | re.MULTILINE)
        for h in reversed(hits):
            val = int(h)
            if 0 <= val <= 99999:
                return val

    return None


def majority_vote(answers: list[int | None]) -> int | None:
    valid = [a for a in answers if a is not None]
    if not valid:
        return None
    return Counter(valid).most_common(1)[0][0]


def score_candidate(text: str, answer: int | None, passed: int, failed: int) -> float:
    """Lightweight verifier-style ranking for 7B candidates."""
    if answer is None:
        return -1.0 - 0.2 * failed

    score = 1.0
    if "\\boxed{" in text:
        score += 0.35
    if re.search(r'(therefore|hence|thus|so the answer is)', text, re.IGNORECASE):
        score += 0.25
    if passed > 0:
        score += 0.30 + 0.05 * passed
    if failed > 0:
        score -= 0.35 * failed
    if len(text) > 2500:
        score -= 0.10
    return score


def choose_best_answer(candidates: list[dict]) -> int | None:
    """Choose an answer by weighted vote, then score, then plain majority."""
    valid = [c for c in candidates if c["answer"] is not None]
    if not valid:
        return None

    if USE_SCORING_VOTE:
        weighted = Counter()
        counts = Counter()
        for c in valid:
            weighted[c["answer"]] += max(c["score"], 0.0)
            counts[c["answer"]] += 1
        best_answer, best_weight = max(weighted.items(), key=lambda kv: (kv[1], counts[kv[0]], -kv[0]))
        if counts[best_answer] >= MIN_VALID_ANSWER_COUNT or best_weight >= 2.2:
            return best_answer

    best = max(valid, key=lambda c: (c["score"], c["passed"], -c["failed"]))
    fallback = majority_vote([c["answer"] for c in valid])
    return fallback if fallback is not None else best["answer"]


# ---------------------------------------------------------------------------
# Code execution (for TIR scoring)
# ---------------------------------------------------------------------------

def execute_code_blocks(text: str, timeout: int = CODE_TIMEOUT_SECONDS) -> tuple[int, int, str]:
    """Run python code blocks. Returns (passed, failed, last_output)."""
    blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    if not blocks:
        return 0, 0, ""
    combined = "\n".join(b.strip() for b in blocks)
    try:
        result = subprocess.run(
            [sys.executable, "-c", combined],
            capture_output=True, text=True, timeout=timeout,
        )
        if result.returncode == 0:
            return len(blocks), 0, result.stdout.strip()
        return 0, len(blocks), result.stderr.strip().split('\n')[-1][:200]
    except subprocess.TimeoutExpired:
        return 0, len(blocks), "timeout"
    except Exception as e:
        return 0, len(blocks), str(e)[:200]


# ---------------------------------------------------------------------------
# Smoke tests
# ---------------------------------------------------------------------------

_tests = [
    (r"Therefore $\\boxed{42}$.", 42),
    (r"The answer is $\\boxed{1234}$.", 1234),
    (r"\\boxed{\\boxed{99999}}", 99999),
    (r"We get \\boxed{0}.", 0),
    (r"The remainder is \\boxed{3}. Note: not 7.", 3),
    (r"So the value is 567.", 567),
    (r"answer is 100000", None),
    (r"No answer here.", None),
]

all_ok = True
for text, expected in _tests:
    got = extract_integer(text)
    ok = got == expected
    if not ok:
        all_ok = False
    print(f"  [{'OK' if ok else 'FAIL'}] extract_integer({text[:40]!r}) -> {got!r}  (expected {expected!r})")

print(f"\n{'All tests passed!' if all_ok else 'Some tests FAILED.'}")
print("Answer extraction ready.")


  [OK] extract_integer('Therefore $\\boxed{42}$.') → 42  (expected 42)
  [OK] extract_integer('The answer is $\\boxed{1234}$.') → 1234  (expected 1234)
  [OK] extract_integer('\\boxed{\\boxed{99999}}') → 99999  (expected 99999)
  [OK] extract_integer('We get \\boxed{0}.') → 0  (expected 0)
  [OK] extract_integer('The remainder is \\boxed{3}. Note: not 7.') → 3  (expected 3)
  [OK] extract_integer('So the value is 567.') → 567  (expected 567)
  [OK] extract_integer('answer is 100000') → None  (expected None)
  [OK] extract_integer('No answer here.') → None  (expected None)

All tests passed!
Answer extraction ready.


---
## 3. TIR Solver

Qwen2.5-Math-Instruct is specifically trained for Tool-Integrated Reasoning:
it writes Python, pauses at `\n```output`, expects the execution result injected,
then continues until it reaches `\boxed{integer}`.

We run N solutions in parallel (vLLM batching), each with up to `max_rounds` of
code→execute→inject→continue. Final answer = majority vote over extracted integers.

In [ ]:
import time

# Qwen2.5-Math-Instruct TIR system prompt (from the model card / AIMO2 winning setup)
SYSTEM_PROMPT = (
    "Please integrate natural language reasoning with programs to solve the problem above, "
    "and put your final integer answer within \\boxed{}."
)

_STOP_TIR = ["```output"]
_STOP_DONE = ["<|im_end|>", "<|endoftext|>"]


def _build_initial_prompt(problem: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _run_code_block(text: str) -> str:
    """Execute the last python block in text, return output string (or error)."""
    blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    if not blocks:
        return ""
    combined = "\n".join(b.strip() for b in blocks)
    try:
        res = subprocess.run(
            [sys.executable, "-c", combined],
            capture_output=True, text=True, timeout=CODE_TIMEOUT_SECONDS,
        )
        out = res.stdout.strip() if res.returncode == 0 else res.stderr.strip().split("\n")[-1][:300]
        return out or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: execution timed out"
    except Exception as e:
        return f"Error: {e}"


def _generate_in_chunks(prompts: list[str], params: SamplingParams, chunk_size: int = GENERATION_CHUNK_SIZE):
    """Generate outputs in small batches to keep KV cache under control."""
    outputs = []
    for i in range(0, len(prompts), chunk_size):
        chunk_prompts = prompts[i:i + chunk_size]
        kwargs = {}
        if LORA_REQUEST is not None:
            kwargs["lora_request"] = LORA_REQUEST
        outputs.extend(llm.generate(chunk_prompts, params, **kwargs))
    return outputs


def _needs_tool_output(text: str) -> bool:
    """Heuristic: model likely stopped before ```output and expects execution result."""
    t = text.rstrip()
    return t.endswith("```") and ("```python" in t)


def _answer_only_fallback(problem: str) -> int | None:
    """Final pass when TIR could not produce an extractable integer."""
    prompt = (
        _build_initial_prompt(problem)
        + "\nProvide only the final integer answer in [0, 99999]. No explanation."
    )
    params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=64,
        n=1,
        stop=_STOP_DONE,
    )
    kwargs = {}
    if LORA_REQUEST is not None:
        kwargs["lora_request"] = LORA_REQUEST
    out = llm.generate([prompt], params, **kwargs)[0].outputs[0].text
    return extract_integer(out)


def _state_full_text(state: dict) -> str:
    return state.get("full_text", state["prompt"] + state["text"])


def _rank_state(text: str, passed: int, failed: int) -> float:
    answer = extract_integer(text)
    if answer is not None:
        return score_candidate(text, answer, passed, failed)

    score = 0.10 * passed - 0.25 * failed
    if "```python" in text:
        score += 0.15
    if re.search(r'(let|suppose|therefore|hence|thus)', text, re.IGNORECASE):
        score += 0.05
    if len(text) > 180:
        score += 0.05
    if len(text) > 2500:
        score -= 0.10
    return score


def _prepare_state_for_next_round(state: dict) -> dict:
    state = dict(state)
    if state.get("done"):
        return state
    if _needs_tool_output(state["text"]):
        code_out = _run_code_block(state["text"])
        if code_out.startswith("Error:") or code_out == "timeout":
            state["failed"] += 1
        else:
            state["passed"] += 1
        state["prompt"] = state["prompt"] + state["text"] + "```output\n" + code_out + "\n```\n"
    else:
        state["prompt"] = state["prompt"] + state["text"]
    state["text"] = ""
    return state


def _build_final_candidates(states: list[dict]) -> list[dict]:
    candidates = []
    for state in states:
        full = _state_full_text(state)
        answer = extract_integer(full)
        candidates.append({
            "answer": answer,
            "text": full,
            "passed": state.get("passed", 0),
            "failed": state.get("failed", 0),
            "score": score_candidate(full, answer, state.get("passed", 0), state.get("failed", 0)),
        })
    return candidates


def solve_majority(problem: str, n: int = SOLUTIONS_PER_PROBLEM, max_rounds: int = MAX_ROUNDS) -> tuple[int | None, list[int | None]]:
    """Solve a problem using TIR plus lightweight candidate scoring."""
    base_prompt = _build_initial_prompt(problem)

    states = [
        {
            "prompt": base_prompt,
            "text": "",
            "done": False,
            "passed": 0,
            "failed": 0,
        }
        for _ in range(n)
    ]

    params = SamplingParams(
        temperature=0.7,
        top_p=0.92,
        max_tokens=MAX_TOKENS,
        n=1,
        stop=_STOP_TIR + _STOP_DONE,
    )
    outputs = _generate_in_chunks([base_prompt] * n, params)

    for i, out in enumerate(outputs):
        states[i]["text"] = out.outputs[0].text
        txt = states[i]["text"]
        states[i]["done"] = (extract_integer(txt) is not None) and (not _needs_tool_output(txt))

    for _ in range(1, max_rounds):
        active = [s for s in states if not s["done"]]
        if not active:
            break

        for idx, state in enumerate(active):
            active[idx] = _prepare_state_for_next_round(state)

        prompts = [s["prompt"] for s in active]
        params_cont = SamplingParams(
            temperature=0.5,
            top_p=0.90,
            max_tokens=MAX_TOKENS,
            n=1,
            stop=_STOP_TIR + _STOP_DONE,
        )
        cont_outputs = _generate_in_chunks(prompts, params_cont)

        for state, out in zip(active, cont_outputs):
            state["text"] = out.outputs[0].text
            full_text = _state_full_text(state)
            if extract_integer(full_text) is not None:
                state["done"] = True
            state["full_text"] = full_text

    candidates = _build_final_candidates(states)
    answers = [c["answer"] for c in candidates]
    voted = choose_best_answer(candidates)
    if voted is None:
        voted = _answer_only_fallback(problem)
    return voted, answers


def solve_beam(problem: str, beam_width: int = BEAM_WIDTH, branch_factor: int = BEAM_BRANCH_FACTOR, max_rounds: int = MAX_ROUNDS) -> tuple[int | None, list[int | None]]:
    """Beam-search variant for 7B: keep only the most promising reasoning traces each round."""
    base_prompt = _build_initial_prompt(problem)
    beam = [{"prompt": base_prompt, "text": "", "done": False, "passed": 0, "failed": 0}]
    finished = []

    for round_idx in range(max_rounds):
        expandable = []
        for state in beam:
            if round_idx > 0:
                state = _prepare_state_for_next_round(state)
            if state.get("done"):
                finished.append(state)
            else:
                expandable.append(state)

        if not expandable:
            break

        prompts = []
        parents = []
        for parent_idx, state in enumerate(expandable):
            for _ in range(branch_factor):
                prompts.append(state["prompt"])
                parents.append(parent_idx)

        params = SamplingParams(
            temperature=0.65 if round_idx == 0 else 0.45,
            top_p=0.92 if round_idx == 0 else 0.88,
            max_tokens=MAX_TOKENS,
            n=1,
            stop=_STOP_TIR + _STOP_DONE,
        )
        outputs = _generate_in_chunks(prompts, params)

        expanded = []
        for parent_idx, out in zip(parents, outputs):
            parent = expandable[parent_idx]
            child = dict(parent)
            child["text"] = out.outputs[0].text
            child["full_text"] = _state_full_text(child)
            child["rank_score"] = _rank_state(child["full_text"], child["passed"], child["failed"])
            child["done"] = (extract_integer(child["full_text"]) is not None) and (not _needs_tool_output(child["text"]))
            expanded.append(child)

        expanded.sort(key=lambda s: (s["rank_score"], s["passed"], -s["failed"]), reverse=True)
        beam = expanded[:beam_width]

        finished.extend([state for state in beam if state.get("done")])
        beam = [state for state in beam if not state.get("done")]

        if len(finished) >= beam_width and all(extract_integer(_state_full_text(s)) is not None for s in finished[:beam_width]):
            break

    final_pool = finished + beam
    candidates = _build_final_candidates(final_pool)
    answers = [c["answer"] for c in candidates]
    voted = choose_best_answer(candidates)
    if voted is None:
        voted = _answer_only_fallback(problem)
    return voted, answers


def solve(problem: str, n: int = SOLUTIONS_PER_PROBLEM, max_rounds: int = MAX_ROUNDS) -> tuple[int | None, list[int | None]]:
    if USE_BEAM_SEARCH:
        return solve_beam(problem, beam_width=BEAM_WIDTH, branch_factor=BEAM_BRANCH_FACTOR, max_rounds=max_rounds)
    return solve_majority(problem, n=n, max_rounds=max_rounds)


# Quick smoke test
t0 = time.time()
ans, all_ans = solve("What is $7^2 - 3^2$?", n=4, max_rounds=2)
print(f"Answer: {ans}  (expected 40)")
print(f"All answers: {all_ans}")
print(f"Time: {time.time()-t0:.1f}s")


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Answer: 40  (expected 40)
All answers: [40, 40, 40, 40]
Time: 2.7s


---
## 4. Benchmark on Reference Problems

AIMO3 provides 10 reference problems with known answers.
We run our solver on them to verify correctness before submission.

In [ ]:
import pandas as pd
import json

# Reference problems (10 problems with known answers) — provided by the competition.
REF_PATH = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"

try:
    ref_df = pd.read_csv(REF_PATH)
    print(f"Loaded {len(ref_df)} reference problems")
    print(ref_df[["id", "answer"]].to_string())
except FileNotFoundError:
    ref_df = None
    print("reference.csv not found — benchmark skipped.")
    print(f"Expected at: {REF_PATH}")

In [ ]:
if ref_df is not None:
    ref_results = []

    def _run_reference_eval(label: str, solver_fn):
        rows = []
        t0 = time.time()

        for _, row in ref_df.iterrows():
            pred, all_preds = solver_fn(row["problem"])
            correct = pred == int(row["answer"])
            rows.append({
                "mode": label,
                "id": row["id"],
                "expected": int(row["answer"]),
                "predicted": pred,
                "correct": correct,
                "all": all_preds,
            })
            votes = Counter(a for a in all_preds if a is not None)
            status = "CORRECT" if correct else "WRONG"
            print(f"[{label}][{status}] id={row['id']}  expected={row['answer']}  got={pred}  votes={dict(votes.most_common(3))}")

        elapsed = time.time() - t0
        accuracy = sum(r["correct"] for r in rows) / len(rows)
        print(f"\n{label} accuracy: {accuracy:.0%} ({sum(r['correct'] for r in rows)}/{len(rows)})  in {elapsed:.1f}s")
        print(f"{label} avg time per problem: {elapsed/len(rows):.1f}s")
        return rows, {"mode": label, "accuracy": accuracy, "elapsed": elapsed}

    primary_mode = "beam" if USE_BEAM_SEARCH else "majority"
    if USE_BEAM_SEARCH:
        primary_solver = lambda problem: solve_beam(problem, beam_width=BEAM_WIDTH, branch_factor=BEAM_BRANCH_FACTOR, max_rounds=MAX_ROUNDS)
    else:
        primary_solver = lambda problem: solve_majority(problem, n=SOLUTIONS_PER_PROBLEM, max_rounds=MAX_ROUNDS)

    primary_rows, primary_summary = _run_reference_eval(primary_mode, primary_solver)
    ref_results.extend(primary_rows)
    benchmark_summaries = [primary_summary]

    if RUN_BASELINE_COMPARISON:
        secondary_mode = "majority" if USE_BEAM_SEARCH else "beam"
        if secondary_mode == "beam":
            secondary_solver = lambda problem: solve_beam(problem, beam_width=BEAM_WIDTH, branch_factor=BEAM_BRANCH_FACTOR, max_rounds=MAX_ROUNDS)
        else:
            secondary_solver = lambda problem: solve_majority(problem, n=SOLUTIONS_PER_PROBLEM, max_rounds=MAX_ROUNDS)

        secondary_rows, secondary_summary = _run_reference_eval(secondary_mode, secondary_solver)
        ref_results.extend(secondary_rows)
        benchmark_summaries.append(secondary_summary)

        print("\nBenchmark comparison summary:")
        for summary in benchmark_summaries:
            print(
                f"  {summary['mode']}: accuracy={summary['accuracy']:.0%}, "
                f"time={summary['elapsed']:.1f}s"
            )

        compare_df = pd.DataFrame(ref_results)
        pivot = compare_df.pivot(index="id", columns="mode", values="correct").reset_index()
        pred_pivot = compare_df.pivot(index="id", columns="mode", values="predicted").reset_index()
        merged = pivot.merge(pred_pivot, on="id", suffixes=("_correct", "_pred"))

        if {"beam", "majority"}.issubset(set(compare_df["mode"])):
            beam_better = ((merged.get("beam_correct", False) == True) & (merged.get("majority_correct", False) != True)).sum()
            majority_better = ((merged.get("majority_correct", False) == True) & (merged.get("beam_correct", False) != True)).sum()
            tie = (merged.get("beam_correct", False) == merged.get("majority_correct", False)).sum()
            accuracy_delta = next(s["accuracy"] for s in benchmark_summaries if s["mode"] == "beam") - next(s["accuracy"] for s in benchmark_summaries if s["mode"] == "majority")

            comparison_summary = {
                "beam_wins": int(beam_better),
                "majority_wins": int(majority_better),
                "ties": int(tie),
                "beam_minus_majority_accuracy": accuracy_delta,
            }
            print("\nHead-to-head summary:")
            print(f"  beam wins: {comparison_summary['beam_wins']}")
            print(f"  majority wins: {comparison_summary['majority_wins']}")
            print(f"  ties: {comparison_summary['ties']}")
            print(f"  beam - majority accuracy: {comparison_summary['beam_minus_majority_accuracy']:+.1%}")
        else:
            comparison_summary = None
    else:
        compare_df = pd.DataFrame(ref_results)
        comparison_summary = None

    compare_df.to_csv("results/reference_benchmark.csv", index=False)
    with open("results/reference_benchmark.json", "w") as f:
        json.dump(ref_results, f, indent=2, default=str)
    with open("results/reference_benchmark_summary.json", "w") as f:
        json.dump(benchmark_summaries, f, indent=2, default=str)
    if comparison_summary is not None:
        with open("results/reference_benchmark_comparison.json", "w") as f:
            json.dump(comparison_summary, f, indent=2, default=str)
        print("Saved to results/reference_benchmark_comparison.json")
    print("Saved to results/reference_benchmark.csv")
    print("Saved to results/reference_benchmark.json")
    print("Saved to results/reference_benchmark_summary.json")
else:
    print("Skipped — no reference file.")


Skipped — no reference file.


---
## 5. Kaggle Submission Loop

The AIMO3 API serves problems one-by-one via `kaggle_evaluation`.  
Our `predict` function is called for each problem and must return an integer 0–99999.

**Scoring:**
- Both predictions correct → 1.0  
- One correct → 0.5  
- Neither → 0.0

In [ ]:
import kaggle_evaluation.aimo_inference_server

def predict(problem: str) -> int:
    """Called by the Kaggle API for each problem. Must return an integer 0–99999."""
    answer, _ = solve(problem, n=SOLUTIONS_PER_PROBLEM, max_rounds=MAX_ROUNDS)
    return int(answer) if answer is not None else 0

inference_server = kaggle_evaluation.aimo_inference_server.AIIMOInferenceServer(predict)

if os.path.exists("/kaggle/working"):
    inference_server.serve()
else:
    # Local dev: test against the placeholder test.csv
    inference_server.run_local_gateway(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"
    )